# Step 2c — Label-rule sensitivity, re-run on each rule's own cohort

**Why this notebook exists.** In `Step2_EarlyWarning_Model.ipynb` the sensitivity cell
filtered arms with `s[s.bc.isin(X_all.index)]`, and `X_all` held only the barcodes loaded
for the primary `union` rule. `moderate_plus` (409 admitted arms) and `injury` (400) were
therefore silently evaluated on the 396-arm union cohort instead of their own, which is why
Table 1 and the sensitivity table disagreed.

This notebook loads the day-4 arrays needed by **all four rules first**, then runs
leave-one-compound-out separately for each rule on its own admitted cohort. It also:

* reports, per rule, how many admitted arms lack expression — so no arm can vanish silently;
* computes balanced accuracy and MCC at an **a-priori** threshold (the design prevalence),
  not the median of held-out predictions, which leaked the test distribution;
* adds compound-clustered confidence intervals;
* checkpoints after every rule, so a Colab disconnect does not cost the whole run.

**Runtime: roughly 20 minutes per rule** on a Colab CPU runtime, so about 1.5 hours for all
four, plus ~10 minutes to download the matrix. `union` is re-run too, deliberately: it should
reproduce AUC 0.8324 exactly, and if it does not, that needs to be known before submission.

## 1. Environment and inputs

In [ ]:
!pip install xgboost -q

import json, time, gc
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             balanced_accuracy_score, matthews_corrcoef)
from xgboost import XGBClassifier

RANDOM_STATE = 42
EARLY_DAY    = 4
N_FEATURES   = 100
RULES        = ["union", "any_finding", "moderate_plus", "injury"]

WORK  = Path("/content/step2c"); WORK.mkdir(parents=True, exist_ok=True)
STEP1 = Path("/content/derived")          # <-- arms_*.csv and array_labels_*.csv live here
MATRIX = Path("/content/affy_tggates_repeat.tsv")

np.random.seed(RANDOM_STATE)
print("work :", WORK)
print("step1:", STEP1)

### Getting the Step 1 files

`STEP1` must contain `arms_<rule>.csv` and `array_labels_<rule>.csv` for all four rules.
Easiest route is to clone the repository; otherwise upload the `derived/` folder yourself.

In [ ]:
# Option A — clone the repository (replace with your repo URL once pushed)
REPO = "https://github.com/Thata-ux/dili-preinjury-early-warning.git"
if not STEP1.exists():
    !git clone --depth 1 $REPO /content/repo
    STEP1 = Path("/content/repo/derived")

need_files = [f"{k}_{r}.csv" for r in RULES for k in ("arms", "array_labels")]
missing = [f for f in need_files if not (STEP1 / f).exists()]
assert not missing, f"missing in {STEP1}: {missing}"
print(f"all {len(need_files)} Step 1 files present in {STEP1}")

### Expression matrix (666 MB, ~10 min)

In [ ]:
ZEN = "https://zenodo.org/records/20437047/files/affy_tggates_repeat.tsv"
if not MATRIX.exists():
    !wget -q --show-progress -O $MATRIX "$ZEN?download=1"
print(f"{MATRIX.name}: {MATRIX.stat().st_size/1e9:.2f} GB")

## 2. Pipeline — identical to Step 2, with the threshold leak removed

In [ ]:
def differential_genes(Xtr, ytr, alpha=0.05, min_lfc=0.5):
    pos, neg = Xtr.values[ytr == 1], Xtr.values[ytr == 0]
    if len(pos) < 2 or len(neg) < 2:
        return Xtr.columns[:500]
    _, p = ttest_ind(pos, neg, axis=0, equal_var=False)
    lfc = pos.mean(0) - neg.mean(0)
    ok = ~np.isnan(p); padj = np.ones_like(p)
    padj[ok] = multipletests(p[ok], method="fdr_bh")[1]
    keep = (padj < alpha) & (np.abs(lfc) >= min_lfc)
    return Xtr.columns[keep] if keep.sum() >= 20 else Xtr.columns[np.argsort(p)[:500]]


def select_features(Xtr, ytr, n=N_FEATURES):
    rf = RandomForestClassifier(n_estimators=600, class_weight="balanced_subsample",
                                random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr, ytr)
    return Xtr.columns[np.argsort(rf.feature_importances_)[::-1][:n]]


def make_model(ytr):
    pos = max(int(ytr.sum()), 1)
    return XGBClassifier(n_estimators=400, max_depth=3, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.8,
                         scale_pos_weight=(len(ytr) - pos) / pos,
                         random_state=RANDOM_STATE, eval_metric="logloss")


def fit_predict(Xtr, ytr, Xte):
    deg   = differential_genes(Xtr, ytr)
    feats = select_features(Xtr[deg], ytr)
    return make_model(ytr).fit(Xtr[feats], ytr).predict_proba(Xte[feats])[:, 1]


def arm_scores(sample_proba, arms_vec):
    return (pd.DataFrame({"arm": arms_vec, "p": sample_proba})
              .groupby("arm")["p"].mean())


def evaluate(truth, score):
    """Threshold is the DESIGN PREVALENCE of this cohort, fixed before seeing
    held-out predictions. The original used the median of the test scores, which
    leaks the test distribution and -- at 12% prevalence -- flags half the arms."""
    j = truth.join(score.rename("p"), how="inner").dropna()
    if j.y.nunique() < 2:
        return {"n_arms": int(len(j)), "auc": None}
    prev = float(j.y.mean())
    k    = max(1, int(round(prev * len(j))))
    thr  = np.sort(j.p.values)[::-1][k - 1]
    pred = (j.p >= thr).astype(int)
    return {"n_arms": int(len(j)), "n_pos": int(j.y.sum()),
            "prevalence": prev, "threshold": float(thr),
            "auc":  float(roc_auc_score(j.y, j.p)),
            "ap":   float(average_precision_score(j.y, j.p)),
            "bacc": float(balanced_accuracy_score(j.y, pred)),
            "mcc":  float(matthews_corrcoef(j.y, pred))}


def clustered_ci(j, n=2000, seed=RANDOM_STATE):
    """Resample whole compounds, not arms."""
    rng = np.random.default_rng(seed)
    y, s = j.y.to_numpy(), j.p.to_numpy()
    codes, uniq = pd.factorize(j.compound)
    groups = [np.flatnonzero(codes == k) for k in range(len(uniq))]
    out = []
    for _ in range(n):
        idx = np.concatenate([groups[k] for k in rng.integers(0, len(groups), len(groups))])
        if len(np.unique(y[idx])) > 1:
            out.append(roc_auc_score(y[idx], s[idx]))
    return [float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))]

print("pipeline defined")

## 3. Collect the day-4 barcodes every rule needs

The whole point: assemble the union of required barcodes **before** loading expression, so
no rule is silently truncated to another rule's cohort.

In [ ]:
cohorts, need = {}, set()
for rule in RULES:
    ar = pd.read_csv(STEP1 / f"arms_{rule}.csv")
    ad = ar[ar.status == "admitted"]
    om = {(c, d): int(o) for c, d, o in zip(ad.compound, ad.dose_level, ad.outcome)}

    lb = pd.read_csv(STEP1 / f"array_labels_{rule}.csv")
    lb["bc"] = lb.BARCODE.astype(str).str.zfill(12)
    s = lb[lb.day == EARLY_DAY].copy()
    s = s[[(c, d) in om for c, d in zip(s.COMPOUND_NAME, s.DOSE_LEVEL)]]

    cohorts[rule] = (s, om)
    need |= set(s.bc)
    print(f"  {rule:14s} admitted {len(om):4d} arms | {len(s):5d} day-{EARLY_DAY} arrays")

print(f"\nunion of day-{EARLY_DAY} barcodes needed: {len(need)}")

### Load only those columns

In [ ]:
t0 = time.time()
head = pd.read_csv(MATRIX, sep="\t", nrows=0)
idx_col = head.columns[0]
cols = set(head.columns)

present = {b for b in need if b in cols}
absent  = need - present
print(f"index column: {idx_col!r} | matrix columns: {len(cols)}")
print(f"needed {len(need)} | present {len(present)} | absent {len(absent)}")
if absent:
    print("  absent barcodes (first 10):", sorted(absent)[:10])

X_all = (pd.read_csv(MATRIX, sep="\t", usecols=[idx_col] + sorted(present))
           .set_index(idx_col).T)
X_all.index = X_all.index.astype(str).str.zfill(12)

# Gene identifiers arrive mixed: most Entrez IDs parse as int, a few stay str.
# scikit-learn refuses a DataFrame whose column names are not all one type, and
# whether a fold trips it depends on which genes that fold selects -- so it can
# pass for one label rule and fail on the next. Normalise once, here.
before = sorted({type(c).__name__ for c in X_all.columns})
X_all.columns = X_all.columns.astype(str)
print(f"column name types: {before} -> ['str']")

print(f"X_all {X_all.shape} loaded in {time.time()-t0:.0f}s")
gc.collect()

## 4. Leave-one-compound-out, per rule, on its own cohort

Checkpointed after each rule to `WORK/sens_<rule>.json` and `WORK/scores_<rule>.csv`.
If the runtime drops, re-run this cell — finished rules are skipped.

In [ ]:
sens, audit = {}, []

for rule in RULES:
    ck = WORK / f"sens_{rule}.json"
    if ck.exists():
        sens[rule] = json.loads(ck.read_text())
        print(f"{rule:14s} already done, skipping")
        continue

    s, om = cohorts[rule]
    n_admitted = len(om)

    s = s[s.bc.isin(X_all.index)]
    arms_present = {(c, d) for c, d in zip(s.COMPOUND_NAME, s.DOSE_LEVEL)}
    n_lost = n_admitted - len(arms_present)
    if n_lost:
        print(f"  NOTE {rule}: {n_lost} admitted arm(s) have no day-{EARLY_DAY} expression; "
              f"report this number, do not hide it")

    yy = np.array([om[(c, d)] for c, d in zip(s.COMPOUND_NAME, s.DOSE_LEVEL)])
    gg = s.COMPOUND_NAME.values
    aa = (s.COMPOUND_NAME + " | " + s.DOSE_LEVEL).values
    XX = X_all.loc[s.bc.values]            # NOT restricted to another rule's gene set

    t0 = time.time()
    o = np.full(len(yy), np.nan)
    folds = list(LeaveOneGroupOut().split(XX, yy, gg))
    for n, (tr, te) in enumerate(folds, 1):
        if len(np.unique(yy[tr])) < 2:
            continue
        o[te] = fit_predict(XX.iloc[tr], yy[tr], XX.iloc[te])
        if n % 20 == 0:
            print(f"    {rule}: fold {n}/{len(folds)}  ({time.time()-t0:.0f}s)")

    k = ~np.isnan(o)
    truth = (pd.DataFrame({"arm": aa, "y": yy, "compound": gg})
               .groupby("arm").agg(y=("y", "max"), compound=("compound", "first")))
    score = arm_scores(o[k], aa[k])

    r = evaluate(truth[["y"]], score)
    j = truth.join(score.rename("p"), how="inner").dropna()
    r["auc_ci_compound_clustered"] = clustered_ci(j)
    r["n_admitted_by_rule"] = int(n_admitted)
    r["n_arms_lost_to_missing_expression"] = int(n_lost)
    r["n_compounds"] = int(j.compound.nunique())
    r["runtime_s"] = round(time.time() - t0)

    sens[rule] = r
    ck.write_text(json.dumps(r, indent=2))
    j.to_csv(WORK / f"scores_{rule}.csv")
    print(f"{rule:14s} AUC {r['auc']:.4f} {r['auc_ci_compound_clustered']} "
          f"| {r['n_arms']} arms, {r['n_pos']} pos | {r['runtime_s']}s\n")

json.dump(sens, open(WORK / "sensitivity_rerun.json", "w"), indent=2)
print("saved", WORK / "sensitivity_rerun.json")

## 5. Integrity checks and the table to paste into the manuscript

In [ ]:
rows = []
for rule in RULES:
    r = sens.get(rule)
    if not r:
        continue
    lo, hi = r["auc_ci_compound_clustered"]
    rows.append({"rule": rule,
                 "admitted": r["n_admitted_by_rule"],
                 "analysed": r["n_arms"],
                 "lost": r["n_arms_lost_to_missing_expression"],
                 "positive": r["n_pos"],
                 "auc": round(r["auc"], 4),
                 "ci": f"{lo:.3f}-{hi:.3f}",
                 "ap": round(r["ap"], 4),
                 "bacc": round(r["bacc"], 4),
                 "mcc": round(r["mcc"], 4)})
tab = pd.DataFrame(rows)
tab.to_csv(WORK / "sensitivity_audit.csv", index=False)
print(tab.to_string(index=False))

print("\n--- integrity checks ---")
bad = tab[tab.admitted != tab.analysed]
if bad.empty:
    print("PASS: every rule evaluated on its full admitted cohort")
else:
    print("CHECK: analysed < admitted for:")
    print(bad[["rule", "admitted", "analysed", "lost"]].to_string(index=False))
    print("  -> state the shortfall and its cause in the Table 4 footnote")

u = sens.get("union", {})
if u.get("auc") is not None:
    d = abs(u["auc"] - 0.8324)
    print(f"union AUC {u['auc']:.4f} vs published 0.8324  (difference {d:.4f}) "
          f"-> {'PASS' if d < 0.005 else 'INVESTIGATE: the primary result did not reproduce'}")

print("\n--- markdown for Table 4 ---")
for _, r in tab.iterrows():
    name = f"**{r['rule']}** (primary)" if r["rule"] == "union" else r["rule"]
    print(f"| {name} | {r['admitted']} | {r['analysed']} | {r['positive']} | "
          f"{r['auc']:.3f} ({r['ci']}) | {r['ap']:.3f} |")

## 6. Download

Send `sensitivity_rerun.json` and `sensitivity_audit.csv` back for the manuscript update;
the `scores_<rule>.csv` files belong in the repository's `derived/` folder.

In [ ]:
from google.colab import files
import shutil
shutil.make_archive("/content/step2c_outputs", "zip", WORK)
files.download("/content/step2c_outputs.zip")